# Hosting Strands Agents with Amazon Bedrock models in Amazon Bedrock AgentCore Runtime

## Overview

In this tutorial we will learn how to host your existing agent, using Amazon Bedrock AgentCore Runtime. We will provide examples using Amazon Bedrock models and non-Bedrock models such as Azure OpenAI and Gemini. We will also showcase Observability through CloudWatch using AWS Opentelemetry Instrumentation and AgentCore Observability.


### Tutorial Details


| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Conversational                                                                   |
| Agent type          | Single                                                                           |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Haiku 4.5                                                      |
| Tutorial components | Hosting agent on AgentCore Runtime. Using Strands Agent and Amazon Bedrock Model |
| Tutorial vertical   | Cross-vertical                                                                   |
| Example complexity  | Easy                                                                             |
| SDK used            | Amazon BedrockAgentCore Python SDK and boto3                                     |

### Tutorial Architecture

In this tutorial we will describe how to deploy an existing agent to AgentCore runtime. 

For demonstration purposes, we will  use a Strands Agent using Amazon Bedrock models

In our example we will use a very simple agent with two tools: `get_weather` and `get_time`. 

<div style="text-align:left">
    <img src="images/architecture_runtime.png" width="50%"/>
</div>

### Tutorial Key Features

* Hosting Agents on Amazon Bedrock AgentCore Runtime
* Using Amazon Bedrock models
* Using Strands Agents
* Amazon CloudWatch GenAI Observability


## Prerequisites

To execute this tutorial you will need:
* Python 3.10+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* Strands Agents
* Docker running
* Amazon CloudWatch Access
* Enable [transaction search](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/Enable-TransactionSearch.html) on Amazon CloudWatch. 


In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet
!pip install --force-reinstall -U -r requirements-dev.txt --quiet

## Creating your agents and experimenting locally

Before we deploy our agents to AgentCore Runtime, let's develop and run them locally for experimentation purposes.

For production agentic applications we will need to decouple the agent creation process from the agent invocation one. With AgentCore Runtime, we will decorate the invocation part of our agent with the `@app.entrypoint` decorator and have it as the entry point for our runtime. Let's first look how each agent is developed during the experimentation phase.


In [ ]:
%%writefile strands_claude.py
"""Simple strands agent demo"""

# pylint disable:W0718

import os
from typing import Optional

from ddgs import DDGS
from strands import Agent, tool
from strands.models import BedrockModel
# from strands.telemetry import StrandsTelemetry
from strands_tools import calculator
from bedrock_agentcore.runtime import BedrockAgentCoreApp, RequestContext


MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
REGION = os.getenv("AWS_REGION", "us-east-1")
SYSTEM_PROMPT = """
You are the Culinary Assistant, a sophisticated restaurant recommendation assistant.
PURPOSE:
- Help users discover restaurants based on their preferences
- Remember user preferences throughout the conversation
- Provide upto 3 personalized dining recommendations
- Include street address for the restaurant recommendation

You have access to a Memory tool that enables you to:
- Retrieve previously stored information to personalize recommendations

You have access to a web search tool that enables you to:
- Retrieve street address for the recommended restaurant
"""

# # Initialize Strands telemetry
# strands_telemetry = StrandsTelemetry()
# strands_telemetry.setup_otlp_exporter()

# Initialize agent application wrapper
app = BedrockAgentCoreApp()


@tool
def web_search(query: str) -> str:
    """
    Search the web for information using DuckDuckGo.
    Args:
        query: The search query
    Returns:
        A string containing the search results
    """
    try:
        ddgs = DDGS()
        results = ddgs.text(query, max_results=3)
        formatted_results = []
        for i, result in enumerate(results, 1):
            formatted_results.append(
                f"{i}. {result.get('title', 'No title')}\n"
                f"   {result.get('body', 'No summary')}\n"
                f"   Source: {result.get('href', 'No URL')}\n"
            )
        return "\n".join(formatted_results) if formatted_results else "No results found."
    except Exception as e:
        return f"Error searching the web: {str(e)}"


@tool
def weather():
    """ Get weather """
    return "sunny"


def initialize_agent(request_ctx: RequestContext):
    """Initialize the agent with  tools"""
    print(request_ctx)

    model = BedrockModel(
        model_id=MODEL_ID,
    )
    agent = Agent(
        tools=[web_search, calculator, weather],
        model=model,
        system_prompt=SYSTEM_PROMPT
    )
    return agent


@app.entrypoint
def strands_agent_bedrock(payload, context: Optional[RequestContext] = None):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    print("User input:", user_input)

    agent = initialize_agent(context)
    response = agent(user_input)
    return response.message['content'][0]['text']


if __name__ == "__main__":
    app.run()


## Deploying the agent to AgentCore Runtime

The `CreateAgentRuntime` operation supports comprehensive configuration options, letting you specify container images, environment variables and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent. 

**Note:** Operations best practice is to package code as container and push to ECR using CI/CD pipelines and IaC

In this tutorial can will the Amazon Bedrock AgentCode Python SDK to easily package your artifacts and deploy them to AgentCore runtime.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "ac_runtime_strands_obsy_demo"
response = agentcore_runtime.configure(
    entrypoint="strands_claude.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    memory_mode='NO_MEMORY'
)
response

### Launching agent to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

In [ ]:
# !agentcore launch \
#     --env BEDROCK_MODEL_ID=us.anthropic.claude-sonnet-4-5-20250929-v1:0 \
#     --env AWS_GENAI_CONTENT_EXTRACTION_OPT_OUT=true \
#     --env AWS_AGENTIC_INSTRUMENTATION=disabled \
#     --env AWS_AGENTIC_INSTRUMENTATION_OPT_IN=false \
#     --env AGENT_OBSERVABILITY_ENABLED=true

!agentcore launch \
    --env BEDROCK_MODEL_ID=us.anthropic.claude-sonnet-4-5-20250929-v1:0 \
    --env AWS_GENAI_CONTENT_EXTRACTION_OPT_OUT=true \
    --env OTEL_SEMCONV_STABILITY_OPT_IN=gen_ai_span_attributes_only


### Checking for the AgentCore Runtime Status
Now that we've deployed the AgentCore Runtime, let's check for it's deployment status

In [ ]:
!agentcore status

### Invoking AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload


In [ ]:
!agentcore invoke '{"prompt": "recommend vegan restaurants in Boston"}'

## Cleanup (Optional)

Let's now clean up the AgentCore Runtime created

In [ ]:
!agentcore destroy --delete-ecr-repo --force --dry-run

### Congratulations!